# Certify the fault distance of circuits

The fault distance of a noisy circuit is the smallest number of faults that flip a logical observable without firing any detectors.
Computing fault distance is expensive in general.
`circuits.get_fault_distance` solves this problem by constructing an auxiliary CSS code whose distance is equal to the fault distance of the original circuit, and uses the Brouwer-Zimmermann algorithm to compute the auxiliary code distance, thereby avoiding a brute-force enumeration of fault combinations.
See `help(circuits.get_fault_distance)` for additional information.

The examples below certify a surface-code memory experiment and a flagged Steane-state preparation circuit.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim

from qldpc import circuits, codes

## Certify the fault distance of a memory experiment

Here we compute the fault distance of an X- and Z-basis memory experiment for a distance-5 surface code with five rounds of syndrome extraction.
Stim's graphlike search for undetectable errors provides a cross-check for this example because every circuit fault in this example flips at most two detectors.
Unlike Stim's graphlike search, however, `circuits.get_fault_distance` does not require graphlike faults.

In [3]:
code = codes.SurfaceCode(5)
noise_model = circuits.DepolarizingNoiseModel(1e-3)

for basis in ("X", "Z"):
    # one logical memory cycle = code.get_distance() rounds of syndrome extraction
    # you can replace logical_memory_cycle with another memory circuit
    initialization, logical_memory_cycle, readout, *_ = circuits.get_memory_experiment_parts(
        code, basis=basis, num_rounds=code.get_distance()
    )

    # keep initialization and readout noiseless, to certify only the inner QEC cycle
    noisy_circuit = initialization + noise_model.noisy_circuit(logical_memory_cycle) + readout
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    graphlike_distance = len(noisy_circuit.shortest_graphlike_error())
    assert fault_distance == graphlike_distance
    print(
        f"{basis}-basis memory: fault distance {fault_distance}, "
        f"Stim graphlike distance {graphlike_distance}"
    )

X-basis memory: fault distance 5, Stim graphlike distance 5
Z-basis memory: fault distance 5, Stim graphlike distance 5


## Certify the fault distance of a state-preparation circuit

We can prepare a $|0_L\rangle$ state of the Steane code with a circuit from the [state-preparation notebook](5_state_preparation.ipynb).
The final qubit in this circuit is measuerd out to detect logical-X faults.

In [4]:
code = codes.SteaneCode()
state_prep_circuit = stim.Circuit("""
    # non-fault-tolerant state preparation
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")

`circuits.get_state_prep_diagnostic_circuit` appends (a) a detector for flag qubit, (b) noiseless final stabilizer measurements, and (c) a noiseless logical-Z observable.
The fault distance of the state preparation circuit is then the fewest faults that cause an undetectable error in the final logical observable.

In [5]:
diagnostic_circuit, _ = circuits.get_state_prep_diagnostic_circuit(
    code,
    state_prep_circuit,
    add_flags=True,
)
noisy_circuit = noise_model.noisy_circuit(diagnostic_circuit)
fault_distance = circuits.get_fault_distance(noisy_circuit)
print(f"State-preparation undetected logical fault distance: {fault_distance}")

State-preparation undetected logical fault distance: 3
